# Baseline

**Autor:** jd · **Datum:** 2026-09-30 · **Issue:** #27

**Ziel:** Referenzwerte für Meilenstein 1, gegen die alle späteren Ansätze antreten. Bewusst
**kein** begründeter Ansatz (siehe #11/#54): eine Untergrenze (Dummy) und ein einfaches Modell,
das Crop und Stage nativ gemeinsam vorhersagt (Random Forest). Ablauf wie in
`docs/modelle/ansatz-entwickeln.md`: `tune()` auf den CV-Folds, dann einmal `run()`.

**Aufbau:** Teil A erzeugt die Läufe (landen in MLflow), Teil B belegt jede Erkenntnis mit einem
Plot. Die wichtigsten Plots werden für die Doku gespeichert (`docs/modelle/img/`).

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

from awp2.config import MODEL_DOCS_FIGURES_DIR, SEED
from awp2.data import valid_combinations
from awp2.evaluation import plot_confusion_matrices
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.plots import (
    plot_class_recall,
    plot_metric_comparison,
    plot_top_confusions,
    plot_tuning_candidates,
    save_doc_figure,
)

# A · Läufe

### Untergrenze: immer die häufigste Klasse

In [ ]:
dummy = run(
    DummyClassifier(strategy="most_frequent"),
    RunConfig(
        name="dummy_most_frequent",
        approach="baseline",
        description="Untergrenze: immer die häufigste Kultur und das häufigste Stadium.",
    ),
)

### Random Forest: Einstellungen per Cross-Validation suchen

Nur auf dem Train-Teil (5 Folds). Kleines Raster: Tiefe, Blattgröße, Anteil der Features je Split.

In [ ]:
tuned = tune(
    RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
    TuneConfig(
        name="rf_search",
        approach="baseline",
        description="Random Forest (Crop und Stage gemeinsam): Tiefe, Blattgröße, max_features.",
        param_grid={
            "max_depth": [None, 20],
            "min_samples_leaf": [1, 3],
            "max_features": ["sqrt", 0.3],
        },
    ),
)

### Bester Random Forest: einmal auf der Validierung bewerten

In [ ]:
rf = run(
    tuned.best_model,
    RunConfig(
        name="rf_baseline",
        approach="baseline",
        description="Bester Random Forest aus rf_search, einmal auf der Validierung bewertet.",
        tuning_run=tuned.run_id,
    ),
)

# B · Erkenntnisse

## 1. Der Random Forest liegt weit über dem Zufallsniveau

Der Dummy erreicht genau das Zufallsniveau der Balanced Accuracy (1/5 Kulturen, 1/6 Stadien).
Der Random Forest trifft Kultur und Stadium einzeln zu fast 90 %; *beides zugleich* richtig
(„kombiniert“) ist spürbar schwerer.

In [ ]:
ax = plot_metric_comparison({"Dummy": dummy.metrics, "Random Forest": rf.metrics})
ax.set_title("Baseline: Dummy vs. Random Forest")
save_doc_figure(ax.figure, "baseline_vergleich", MODEL_DOCS_FIGURES_DIR)

## 2. Mehr Bänder pro Entscheidung helfen – und die CV schätzt ehrlich

Alle 8 Kandidaten der Suche, eingefärbt nach `max_features`: Mit 30 % der Features je Split
(`0.3`) liegen alle Kandidaten deutlich vor `sqrt` (≈ 11 von 139 Features). Die Information steckt
also in vielen, stark korrelierten Bändern. Die gestrichelte Linie ist der Score des besten
Kandidaten auf der Validierung – er liegt im Streubereich der CV, der Holdout wurde also nicht
überschätzt.

In [ ]:
ax = plot_tuning_candidates(
    tuned.candidates, color_by="max_features", validation_score=rf.metrics.bacc_combined
)
ax.set_title("Tuning Random Forest: CV-Score je Kandidat (BAcc kombiniert)")
save_doc_figure(ax.figure, "baseline_tuning", MODEL_DOCS_FIGURES_DIR)

## 3. Verwechselt werden Mais ↔ Soja und benachbarte Stadien

Die Confusion Matrices (zeilenweise normiert) und die häufigsten Verwechslungen zeigen dasselbe
Muster: Bei der Kultur liegen die Fehler vor allem zwischen **Mais und Soja** – beides
Sommerkulturen mit ähnlicher Saison. Beim Stadium verwechselt das Modell vor allem
**benachbarte Stadien**, deren Spektren sich naturgemäß ähneln – und Mature_Senesc mit
Emerge_VEarly: wenig grüne Vegetation, viel Boden, trotz entgegengesetzter Saisonphase.

In [ ]:
fig = plot_confusion_matrices(rf.y_val, rf.y_pred)
save_doc_figure(fig, "baseline_confusion", MODEL_DOCS_FIGURES_DIR)

In [ ]:
fig = plot_top_confusions(rf.y_val, rf.y_pred)
save_doc_figure(fig, "baseline_verwechslungen", MODEL_DOCS_FIGURES_DIR)

## 4. Am schwächsten: Mature_Senesc

Recall je Klasse (Anteil der Zeilen einer Klasse, die richtig erkannt werden). Bei den Kulturen
liegen alle nah beieinander; beim Stadium fällt **Mature_Senesc** ab – es wird mit frühen und
kritischen Stadien verwechselt (siehe oben).

In [ ]:
fig = plot_class_recall(rf.y_val, rf.y_pred)
save_doc_figure(fig, "baseline_recall", MODEL_DOCS_FIGURES_DIR)

## 5. Einige Vorhersagen sind unmögliche Kombinationen

Der Random Forest sagt Kultur und Stadium zwar gemeinsam, aber ohne ihre Abhängigkeit vorher.
Ein kleiner Teil der Vorhersagen sind Paare, die es im Training nie gibt:

In [ ]:
valid = {(c.crop, c.stage) for c in valid_combinations(rf.y_val)}
predicted = pd.Series(list(zip(rf.y_pred["Crop"], rf.y_pred["Stage"], strict=True)))
invalid = predicted[~predicted.isin(valid)].value_counts()
print(f"Anteil unmöglicher Paare: {rf.metrics.invalid_combinations:.1%}")
invalid.rename("Anzahl").to_frame()

# Fazit

| | BAcc Kultur | BAcc Stadium | BAcc kombiniert |
| --- | --- | --- | --- |
| Dummy | 0.200 | 0.167 | 0.044 |
| Random Forest | 0.890 | 0.870 | **0.743** |

- **Messlatte für alle weiteren Ansätze: BAcc kombiniert 0.743.**
- Mais ↔ Soja und benachbarte Stadien sind die Schwachstellen → Ansatzpunkte für Features
  (z. B. Red Edge, SWIR) und für die Fehleranalyse in M2.
- Unmögliche Paare und die Verwechslung benachbarter Stadien sprechen dafür, die Abhängigkeit
  zwischen Kultur und Stadium auszunutzen (hierarchischer oder kombinierter Ansatz).

Details: `docs/modelle/ansaetze/baseline.md`, MLflow-Experiment `crop-stage`.